In [1]:
import os
import re
import shutil
from pathlib import Path

# Define root paths relative to the notebook's location
DOCS_DIR = Path("docs")
IMAGES_DIR = DOCS_DIR / "assets" / "images"
BACKUP_DIR = Path("orphaned_images_backup")

# Regex pattern to match markdown image syntax: ![alt](path)
# This captures the alt text in group 1 and the image path/filename in group 2
IMG_REGEX = re.compile(r"!\[(.*?)\]\((.*?)\)")

In [2]:
def process_markdown_images(fix_paths=False):
    """
    Crawls markdown files to fix image paths to relative asset targets
    and detects broken/missing images.
    """
    missing_images = set()
    referenced_images = set()
    
    # Walk through all markdown files in the docs directory
    for md_path in DOCS_DIR.rglob("*.md"):
        # Skip assets or themes if any accidentally match
        if "assets" in md_path.parts:
            continue
            
        with open(md_path, "r", encoding="utf-8") as f:
            content = f.read()
            
        matches = IMG_REGEX.findall(content)
        if not matches:
            continue
            
        new_content = content
        file_updated = False
        
        for alt_text, img_path in matches:
            # Extract just the raw filename (e.g., 'page_4_image_1_v2.jpg')
            img_name = Path(img_path).name
            referenced_images.add(img_name)
            
            # Check if the file actually exists in assets/images
            actual_image_path = IMAGES_DIR / img_name
            if not actual_image_path.exists():
                missing_images.add((md_path.name, img_name))
            
            if fix_paths:
                # Calculate the relative path from the current MD file to assets/images
                # e.g., from 'docs/Unit 1/file.md' to 'docs/assets/images/' is '../assets/images/'
                relative_dir = os.path.relpath(IMAGES_DIR, md_path.parent)
                correct_img_path = os.path.join(relative_dir, img_name).replace("\\", "/")
                
                # Replace the old image reference with the clean relative one
                old_markdown_link = f"![{alt_text}]({img_path})"
                new_markdown_link = f"![{alt_text}]({correct_img_path})"
                
                if old_markdown_link != new_markdown_link:
                    new_content = new_content.replace(old_markdown_link, new_markdown_link)
                    file_updated = True
                    
        if fix_paths and file_updated:
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(new_content)
            print(f"✔️ Updated paths in: {md_path.relative_to(DOCS_DIR)}")
            
    # Print results for broken links
    print("\n--- BROKEN / MISSING IMAGES REPORT ---")
    if missing_images:
        print(f"❌ Found {len(missing_images)} broken image link(s):")
        for md_file, missing_img in sorted(missing_images):
            print(f"   - In '{md_file}': Graphic '{missing_img}' is missing from assets folder.")
    else:
        print("✅ No broken image links detected! All referenced images exist.")
        
    return referenced_images

In [3]:
def clean_orphaned_images(referenced_images):
    """
    Compares physical files against referenced images and moves unused ones to a backup folder.
    """
    if not IMAGES_DIR.exists():
        print("Error: Images directory does not exist.")
        return

    # Gather all images currently inside the folder
    physical_images = {f.name for f in IMAGES_DIR.iterdir() if f.is_file()}
    
    # Orphaned images are those present physically but never linked in code
    orphaned_images = physical_images - referenced_images
    
    print("\n--- ORPHANED IMAGES REPORT ---")
    if orphaned_images:
        print(f"📦 Found {len(orphaned_images)} orphaned image(s). Moving to '{BACKUP_DIR}/'...")
        BACKUP_DIR.mkdir(exist_ok=True)
        
        for img_name in sorted(orphaned_images):
            source = IMAGES_DIR / img_name
            destination = BACKUP_DIR / img_name
            shutil.move(str(source), str(destination))
            print(f"   -> Moved: {img_name}")
        print("📁 Cleanup complete.")
    else:
        print("🎉 Zero orphaned images found! Your assets folder is pristine.")

In [4]:
used_images = process_markdown_images(fix_paths=True)
used_images


--- BROKEN / MISSING IMAGES REPORT ---
❌ Found 1 broken image link(s):
   - In '8_3_cosmology.md': Graphic 'image_url_placeholder' is missing from assets folder.


{'image_url_placeholder',
 'page_100_image_2_v2.jpg',
 'page_101_image_2_v2.jpg',
 'page_101_image_8_v2.jpg',
 'page_104_chart_1_v2.jpg',
 'page_106_image_1_v2.jpg',
 'page_107_image_1_v2.jpg',
 'page_10_chart_1_v2.jpg',
 'page_10_image_3_v2.jpg',
 'page_110_image_3_v2.jpg',
 'page_111_image_1_v2.jpg',
 'page_112_image_1_v2.jpg',
 'page_114_chart_1_v2.jpg',
 'page_115_image_1_v2.jpg',
 'page_118_image_2_v2.jpg',
 'page_119_image_1_v2.jpg',
 'page_120_image_1_v2.jpg',
 'page_124_image_1_v2.jpg',
 'page_125_image_1_v2.jpg',
 'page_128_image_1_v2.jpg',
 'page_132_image_1_v2.jpg',
 'page_133_image_1_v2.jpg',
 'page_134_image_1_v2.jpg',
 'page_135_image_1_v2.jpg',
 'page_135_image_2_v2.jpg',
 'page_137_image_1_v2.jpg',
 'page_138_image_1_v2.jpg',
 'page_138_image_2_v2.jpg',
 'page_139_image_1_v2.jpg',
 'page_13_image_1_v2.jpg',
 'page_141_image_2_v2.jpg',
 'page_142_image_1_v2.jpg',
 'page_143_image_1_v2.jpg',
 'page_146_chart_1_v2.jpg',
 'page_147_image_1_v2.jpg',
 'page_148_image_1_v2.jpg

In [5]:
len(used_images)

298

In [6]:
clean_orphaned_images(used_images)


--- ORPHANED IMAGES REPORT ---
📦 Found 249 orphaned image(s). Moving to 'orphaned_images_backup/'...
   -> Moved: img_p10_1.png
   -> Moved: img_p10_2.png
   -> Moved: img_p10_3.png
   -> Moved: img_p11_1.png
   -> Moved: img_p11_2.png
   -> Moved: img_p12_1.png
   -> Moved: img_p12_2.png
   -> Moved: img_p13_1.png
   -> Moved: img_p13_2.png
   -> Moved: img_p13_3.png
   -> Moved: img_p14_1.png
   -> Moved: img_p14_2.png
   -> Moved: img_p14_3.png
   -> Moved: img_p14_4.png
   -> Moved: img_p15_1.png
   -> Moved: img_p15_2.png
   -> Moved: img_p16_1.png
   -> Moved: img_p16_2.png
   -> Moved: img_p16_3.png
   -> Moved: img_p17_1.png
   -> Moved: img_p17_2.png
   -> Moved: img_p17_3.png
   -> Moved: img_p17_4.png
   -> Moved: img_p18_1.png
   -> Moved: img_p18_2.png
   -> Moved: img_p18_3.png
   -> Moved: img_p19_1.png
   -> Moved: img_p19_2.png
   -> Moved: img_p19_3.png
   -> Moved: img_p1_1.png
   -> Moved: img_p1_2.png
   -> Moved: img_p20_1.png
   -> Moved: img_p20_2.png
   -> Mov